This dataset has the tweets from an airline company. Perform the following activities given the above dataset
1. Build a classification model using the airline_sentiment column as the target variable and use the ?text? column as the input text for the model
2. Perform the following preprocessing
    * a. Clean the text
    * b. Create Document term matrix using the TFIDFVectorizer
    * c. Add additional variables or features from the dataset to improve the performance
    * d. Perform lemmatisation or any other form of text cleansing
    * e. Create a feature that counts the number of punctuation marks per each sentence
    * f. Create a feature that counts the length of each tweet
3. Compute metrics such as precision recall and classification for all the three classes
4. Build a feed forward neural network that can perform multi class classification

# Importing the required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
import string

import warnings
warnings.filterwarnings('ignore')


# Reading the tweets

In [ ]:
data = pd.read_csv('Tweets.csv')
data.head()

In [ ]:
data.info()

Creating a tweets dataframe for text and sentiment

In [ ]:
tweets = data[['text','airline_sentiment']]

In [ ]:
tweets.info()

Distribution of Tweets sentiments

In [ ]:
tweets.airline_sentiment.value_counts().plot(kind = 'bar')

Cleaning the text 
* Removing the mentions
* Removing the #
* Removing links

In [ ]:
tweets['text_clean'] = tweets['text'].apply(lambda x : re.sub(r'\@\w*','',x))
tweets['text_clean'] = tweets['text_clean'].apply(lambda x : re.sub(r'\#','',x))
tweets['text_clean'] = tweets['text_clean'].apply(lambda x : re.sub(r'https?:\/\/.*\/\w*','',x))

In [ ]:
tweets.head()

Creating features :
* Punctuation Count
* Word Count

In [ ]:

nlp = spacy.load("en_core_web_sm")


def punctuation_count(sent):
    count = 0
    doc = nlp(sent)
    for token in doc:
        if token.lemma_ in string.punctuation:
            count+=1
    return count

def word_count(sent):
    count = 0
    doc = nlp(sent)
    for token in doc:
        if token.lemma_ != " ":
            count+=1
    return count




In [ ]:
tweets['Punctuation_count'] = tweets.text.apply(lambda x : punctuation_count(x))

In [ ]:
tweets['word_count'] = tweets.text.apply(lambda x : word_count(x))

In [ ]:
def clean_doc(sent):
    doc = nlp(sent)
    lemma_out = [token.lemma_ for token in doc]
    lemma_out = " ".join(lemma_out)
    return lemma_out

tweets["text_clean"] = tweets.text_clean.apply(lambda x: clean_doc(x))

In [ ]:
tweets

# TFIDF Vector

In [ ]:
vec = TfidfVectorizer(stop_words='english')


In [ ]:
vectorized_data = vec.fit_transform(tweets["text_clean"])

In [ ]:
result = pd.DataFrame(
    data=vectorized_data.toarray(),  
    columns=vec.get_feature_names_out()
)
result

In [ ]:
result

Dropping features that are just numbers

In [ ]:
columns_num = []
for column in result.columns:
    if column.isdigit():
        columns_num.append(column)


In [ ]:
result.drop(columns_num,inplace = True,axis=1)
result

Random Forest Classifier on data with Features excluding numbers

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(result, tweets["airline_sentiment"], test_size=0.3)


## Model Building
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier()
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
preds

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,preds)

Random Forest Classifier on data with Features including numbers

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(vectorized_data, tweets["airline_sentiment"], test_size=0.3)


## Model Building
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
accuracy_score(y_test,preds)

Deep Neural Network

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout,Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:
y = pd.get_dummies(tweets["airline_sentiment"])
X_train, X_test, y_train, y_test = train_test_split(result, y, test_size=0.1)
X_test, X_val, y_test, y_val = train_test_split(X_test,y_test,test_size=0.1)


In [ ]:
ANN_model = Sequential()
ANN_model.add(Input(result.shape[1]))
ANN_model.add(Dense(50,activation = 'relu'))
ANN_model.add(Dropout(0.3))
ANN_model.add(Dense(3,activation = 'softmax'))


In [ ]:
ANN_model.compile(metrics='accuracy',loss='categorical_crossentropy',optimizer=Adam(learning_rate= 0.0001))

In [ ]:
es = EarlyStopping(monitor='val_loss',patience=2,mode='min')

In [ ]:
ANN_model.fit(x = X_train,y = y_train,epochs = 25,validation_data=(X_test,y_test),callbacks=[es])

In [ ]:
pred = ANN_model.predict(X_val)
ANN_model.evaluate(X_val,y_val)

In [ ]:
str = "I am very happy that my assignment is complete"
str = clean_doc(str)
str = vec.transform([str]) 

In [ ]:
test_str = pd.DataFrame(
    data=str.toarray(),  
    columns=vec.get_feature_names_out()
)
test_str.drop(columns_num,inplace = True,axis=1)
test_str

In [ ]:
classes = list(y.columns)
sentiments = ANN_model.predict(test_str)
classes[np.argmax(sentiments)]

In [ ]:
clf.predict(str)